# Análisis Exploratorio de Datos
**Dataset:** [Nombre del dataset]  
**Fecha:** [Fecha]  
**Objetivo principal:** [Describir objetivo]

---

## PARTE I - PREPARACIÓN

### 0. Importar librerías

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline
```

### 1. Establecer objetivos

**Objetivos del análisis:**
- 
- 
- 

**Preguntas clave a responder:**
1. 
2. 
3. 

---

### 2. Cargar y explorar datos

```python
# Cargar datos
df = pd.read_csv('ruta/al/archivo.csv')

# Primera exploración
print(f"Dimensiones: {df.shape}")
print(f"\nPrimeras filas:")
df.head()
```

**Ficha del dataset:**
- **Nombre:** 
- **Fuente:** 
- **Número de registros:** 
- **Número de variables:** 
- **Descripción:** 

---

### 3. Tipificar los datos



```python
# Información general
df.info()

# Tipos de datos
print("\nTipos de datos:")
print(df.dtypes)

# Identificar tipos de variables
numericas = df.select_dtypes(include=[np.number]).columns.tolist()
categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\nVariables numéricas ({len(numericas)}): {numericas}")
print(f"Variables categóricas ({len(categoricas)}): {categoricas}")
```



Notas sobre tipos de datos:
- 
- 

---

### 4. Priorizar variables

**Variables clave identificadas:**  
-  **Target/Dependiente:**   
-  **Principales independientes:**   
  - 
  - 
-  **Secundarias:** 
  - 

 
- **Variables a descartar (y por qué):**


---

## PARTE II - ANÁLISIS UNIVARIANTE

### 5. Análisis de valores faltantes



```python
# Detectar valores faltantes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Variable': missing.index,
    'Missing': missing.values,
    'Porcentaje': missing_pct.values
}).sort_values('Porcentaje', ascending=False)

print(missing_df[missing_df['Missing'] > 0])

# Visualizar patrones de missings (requiere: pip install missingno)
# import missingno as msno
# msno.matrix(df)
# plt.show()
```



**Decisiones sobre valores faltantes:**
- 
- 

---

### 6. Tendencia central - todas las variables



```python
# Resumen estadístico
df.describe()
```



```python
# Para categóricas
for col in categoricas:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"Moda: {df[col].mode()[0] if len(df[col].mode()) > 0 else 'N/A'}")
```



### 7. Frecuencias - Variables categóricas



```python
# Gráficos de barras para categóricas
fig, axes = plt.subplots(nrows=(len(categoricas)+2)//3, ncols=3, figsize=(15, 5*((len(categoricas)+2)//3)))
axes = axes.flatten()

for i, col in enumerate(categoricas):
    df[col].value_counts().plot(kind='bar', ax=axes[i])
    axes[i].set_title(f'Distribución de {col}')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=45)

# Ocultar ejes vacíos
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()
```



**Observaciones sobre categóricas:**
- 
- 

---

### 8. Posición y rangos - Variables numéricas



```python
# Boxplots
fig, axes = plt.subplots(nrows=(len(numericas)+2)//3, ncols=3, figsize=(15, 5*((len(numericas)+2)//3)))
axes = axes.flatten()

for i, col in enumerate(numericas):
    df.boxplot(column=col, ax=axes[i])
    axes[i].set_title(f'Boxplot de {col}')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()
```



```python
# Cuartiles y rangos
for col in numericas:
    print(f"\n{col}:")
    print(f"  Min: {df[col].min()}")
    print(f"  Q1: {df[col].quantile(0.25)}")
    print(f"  Mediana: {df[col].median()}")
    print(f"  Q3: {df[col].quantile(0.75)}")
    print(f"  Max: {df[col].max()}")
    print(f"  Rango: {df[col].max() - df[col].min()}")
    print(f"  IQR: {df[col].quantile(0.75) - df[col].quantile(0.25)}")
```



---

### 9. Dispersión - Variables numéricas



```python
# Medidas de dispersión
dispersion = pd.DataFrame({
    'Variable': numericas,
    'Std': [df[col].std() for col in numericas],
    'Varianza': [df[col].var() for col in numericas],
    'CV': [(df[col].std()/df[col].mean())*100 if df[col].mean() != 0 else 0 for col in numericas]
})
print(dispersion)
```



**Observaciones sobre dispersión:**
- 
- 

---

### 10. Distribuciones - Variables numéricas



```python
# Histogramas y KDE
fig, axes = plt.subplots(nrows=len(numericas), ncols=2, figsize=(12, 4*len(numericas)))

for i, col in enumerate(numericas):
    # Histograma
    axes[i, 0].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    axes[i, 0].set_title(f'Histograma de {col}')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Frecuencia')
    
    # KDE
    df[col].dropna().plot(kind='density', ax=axes[i, 1])
    axes[i, 1].set_title(f'Densidad de {col}')
    axes[i, 1].set_xlabel(col)

plt.tight_layout()
plt.show()
```



**Observaciones sobre distribuciones:**
- 
- 

---

### 11. Detectar anomalías/outliers



```python
# Método IQR
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

# Análisis de outliers
for col in numericas:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"\n{col}:")
    print(f"  Límite inferior: {lower:.2f}")
    print(f"  Límite superior: {upper:.2f}")
    print(f"  Número de outliers: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")
    
    # Método de desviaciones estándar
    mean = df[col].mean()
    std = df[col].std()
    outliers_std = df[(df[col] < mean - 3*std) | (df[col] > mean + 3*std)]
    print(f"  Outliers por 3σ: {len(outliers_std)} ({len(outliers_std)/len(df)*100:.2f}%)")
```



---

### 12. Tratar outliers



```python
# OPCIÓN 1: Identificar y etiquetar
df['outlier_variable'] = ((df['variable'] < lower) | (df['variable'] > upper)).astype(int)

# OPCIÓN 2: Separar en dataset diferente
df_outliers = df[(df['variable'] < lower) | (df['variable'] > upper)]
df_clean = df[~((df['variable'] < lower) | (df['variable'] > upper))]

# OPCIÓN 3: Eliminar
df = df[~((df['variable'] < lower) | (df['variable'] > upper))]

# OPCIÓN 4: Winsorizar (límite)
from scipy.stats.mstats import winsorize
df['variable_win'] = winsorize(df['variable'], limits=[0.05, 0.05])
```



**Decisiones sobre outliers:**
- 
- 

---

## PARTE III - ANÁLISIS BIVARIANTE

### 13. Revisar prioridades y preguntas

**Recapitulación hasta ahora:**
- **Hallazgos principales:**
  - 
  - 
- **Preguntas respondidas:**
  - 
- **Preguntas pendientes:**
  - 

**Actualización de prioridades:**
- 

---

### 14. Análisis con categóricas importantes

#### 14.1 Categórica vs Categórica



```python
# Variables a analizar
cat_target = 'variable_target'  # Cambiar
cat_features = ['cat1', 'cat2']  # Cambiar

# Tablas de contingencia y gráficos
for cat in cat_features:
    print(f"\n--- {cat_target} vs {cat} ---")
    
    # Tabla cruzada
    ct = pd.crosstab(df[cat], df[cat_target], margins=True)
    print(ct)
    
    # Tabla de proporciones
    print("\nProporciones:")
    print(pd.crosstab(df[cat], df[cat_target], normalize='index'))
    
    # Gráfico de barras agrupado
    pd.crosstab(df[cat], df[cat_target]).plot(kind='bar', figsize=(10, 5))
    plt.title(f'{cat_target} por {cat}')
    plt.xticks(rotation=45)
    plt.legend(title=cat_target)
    plt.tight_layout()
    plt.show()
```



#### 14.2 Categórica vs Numérica



```python
# Variables a analizar
num_features = ['num1', 'num2']  # Cambiar

for num in num_features:
    print(f"\n--- {num} por {cat_target} ---")
    
    # Estadísticas descriptivas por grupo
    print(df.groupby(cat_target)[num].describe())
    
    # Boxplot por categoría
    plt.figure(figsize=(10, 5))
    df.boxplot(column=num, by=cat_target)
    plt.title(f'{num} por {cat_target}')
    plt.suptitle('')
    plt.show()
    
    # Violinplot (alternativa)
    plt.figure(figsize=(10, 5))
    sns.violinplot(data=df, x=cat_target, y=num)
    plt.title(f'Distribución de {num} por {cat_target}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
```



**Observaciones categóricas:**
- 
- 

---

### 15. Análisis con numéricas importantes

#### 15.1 Numérica vs Numérica



```python
# Matriz de correlación
correlacion = df[numericas].corr()
print("Matriz de correlación:")
print(correlacion)

# Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlacion, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

# Correlaciones más fuertes con la variable target (si aplica)
# target_corr = correlacion['target'].sort_values(ascending=False)
# print("\nCorrelaciones con target:")
# print(target_corr)
```

```python
# Pairplot (solo con variables seleccionadas si son muchas)
# variables_pairplot = ['var1', 'var2', 'var3', 'target']  # Seleccionar
# sns.pairplot(df[variables_pairplot], hue='target')  # Si hay variable categórica
# plt.show()
```

```python
# Scatter plots específicos de interés
# plt.figure(figsize=(8, 6))
# plt.scatter(df['var1'], df['var2'], alpha=0.5)
# plt.xlabel('Variable 1')
# plt.ylabel('Variable 2')
# plt.title('Relación entre Variable 1 y Variable 2')
# plt.show()
```



**Observaciones numéricas:**
- 
- 

---

### 16. Recapitulación

**Clasificación de hallazgos:**

**MENSAJES PRINCIPALES (para comunicar):**
1. 
2. 
3. 

**ELEMENTOS A PROFUNDIZAR:**
- 
- 

**HALLAZGOS INTERESANTES PERO NO CRÍTICOS:**
- 
- 

**PREGUNTAS CONTESTADAS:**
- ✓ 
- ✓ 

**PREGUNTAS SIN CONTESTAR:**
- ❌ 
- ❌ 

---

## PARTE IV - ANÁLISIS AVANZADO

### 17. Resolver preguntas pendientes



```python
# Análisis específicos para preguntas pendientes

# Pregunta 1:


# Pregunta 2:


```



---

### 18. Análisis multivariante (3+ variables)



```python
# Ejemplo: Análisis de 3 variables
# Planificación:
# - Variable 1:
# - Variable 2:
# - Variable 3:
# - Tipo de análisis:

# Código:

```



---

### 19. Análisis personalizados



```python
# Variaciones y análisis específicos del dominio


```



---

### 20. Conclusiones finales

**RESUMEN EJECUTIVO:**
- 
- 
- 

**HALLAZGOS PRINCIPALES:**
1. 
2. 
3. 

**RECOMENDACIONES:**
- 
- 

**LÍNEAS DE TRABAJO FUTURO:**
- 
- 

**LIMITACIONES DEL ANÁLISIS:**
- 
- 

---

## ANEXO - Funciones útiles



```python
# Función para análisis rápido de variable
def analisis_rapido(df, variable):
    """Análisis completo de una variable"""
    print(f"=== Análisis de {variable} ===\n")
    
    if df[variable].dtype in ['object', 'category']:
        print("Tipo: Categórica")
        print(f"\nFrecuencias:\n{df[variable].value_counts()}")
        print(f"\nValores únicos: {df[variable].nunique()}")
        
        plt.figure(figsize=(10, 5))
        df[variable].value_counts().plot(kind='bar')
        plt.title(f'Distribución de {variable}')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Tipo: Numérica")
        print(f"\n{df[variable].describe()}")
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Histograma
        axes[0].hist(df[variable].dropna(), bins=30, edgecolor='black')
        axes[0].set_title('Histograma')
        
        # Boxplot
        axes[1].boxplot(df[variable].dropna())
        axes[1].set_title('Boxplot')
        
        # KDE
        df[variable].dropna().plot(kind='density', ax=axes[2])
        axes[2].set_title('Densidad')
        
        plt.tight_layout()
        plt.show()

# Ejemplo de uso:
# analisis_rapido(df, 'nombre_variable')
```



---

**NOTAS FINALES:**
- 
-